# Phase 16C — Selective-prediction (risk-coverage) table for the final model

Standalone notebook that reloads the already-trained final model (`checkpoints/edes_cqr_v1/latest.pt`, produced by `15_ed_es_final_model.ipynb`) and runs **inference only** -- nothing is trained here. It produces the risk-coverage results for the final model reported in the paper (Section 4.5, Table 4).

Run the cells top to bottom with a GPU runtime. Requires `14_ed_es_extraction.ipynb` and `15_ed_es_final_model.ipynb` to be complete first.

## Setup — imports, Drive mount, data splits

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, mean_absolute_error

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
EDES_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/edes', 'frames')
quality_csv_path = os.path.join(PROJECT_ROOT, 'data/full', 'quality_scores.csv')
PRETRAINED_BACKBONE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'simclr_pretrain_v1', 'backbone_only.pt')
FINAL_MODEL_CKPT = os.path.join(PROJECT_ROOT, 'checkpoints', 'edes_cqr_v1', 'latest.pt')  # notebook 15's trained final model

assert os.path.isdir(EDES_CACHE_DIR) and len(os.listdir(EDES_CACHE_DIR)) > 9000, \
    'Run 14_ed_es_extraction.ipynb to completion first.'

df = pd.read_csv(FULL_CSV)
quality_df = pd.read_csv(quality_csv_path)
ef_col = 'EF' if 'EF' in df.columns else [c for c in df.columns if 'ef' in c.lower()][0]
filename_col = 'FileName' if 'FileName' in df.columns else [c for c in df.columns if 'file' in c.lower() and 'name' in c.lower()][0]
HFrEF_THRESHOLD = 40

for col in ['sharpness', 'contrast']:
    lo, hi = quality_df[col].quantile(0.02), quality_df[col].quantile(0.98)
    quality_df[col + '_norm'] = ((quality_df[col] - lo) / max(hi - lo, 1e-6)).clip(0, 1)
quality_df['quality_score'] = quality_df[['sharpness_norm', 'contrast_norm']].mean(axis=1)
df = df.merge(quality_df[[filename_col, 'quality_score']], on=filename_col, how='left')

valid_mask = df[filename_col].apply(lambda f: os.path.exists(
    os.path.join(EDES_CACHE_DIR, (str(f) if str(f).endswith('.avi') else str(f)+'.avi').replace('.avi', '.npy'))))
clean_df = df[valid_mask & df['quality_score'].notna()].reset_index(drop=True)
train_df = clean_df[clean_df['Split'].str.upper() == 'TRAIN'].reset_index(drop=True)
val_df = clean_df[clean_df['Split'].str.upper() == 'VAL'].reset_index(drop=True)
test_df = clean_df[clean_df['Split'].str.upper() == 'TEST'].reset_index(drop=True)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: enable GPU before running this notebook (Runtime > Change runtime type).')

## Dataset class and loaders (num_workers=0 + retry-on-read-failure, for Drive reliability)

In [ ]:
class EchoEDESDataset(Dataset):
    def __init__(self, df, frame_dir, filename_col, ef_col, train=True):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.filename_col = filename_col
        self.ef_col = ef_col
        self.train = train
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.df)

    def _load_pair(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.filename_col])
        if not fname.endswith('.avi'):
            fname = fname + '.avi'
        path = os.path.join(self.frame_dir, fname.replace('.avi', '.npy'))
        pair = np.load(path)
        pair = torch.from_numpy(pair).permute(0, 3, 1, 2).float() / 255.0
        pair = F.interpolate(pair, size=(112, 112), mode='bilinear', align_corners=False)
        return pair, row

    def __getitem__(self, idx):
        # Retry a few times on transient Google Drive read failures (FileNotFoundError /
        # OSError), which happen occasionally under concurrent access even for files that
        # do exist. Falls back to a different random index only after repeated failures on
        # the SAME index, so a genuinely-missing file doesn't silently swap in unrelated data.
        last_err = None
        for attempt in range(5):
            try:
                pair, row = self._load_pair(idx)
                break
            except (FileNotFoundError, OSError) as e:
                last_err = e
                time.sleep(0.5 * (attempt + 1))
        else:
            raise RuntimeError(
                f'Repeated Drive read failure for index {idx} '
                f'({self.df.iloc[idx][self.filename_col]}) after 5 attempts: {last_err}'
            )
        if self.train and torch.rand(1).item() < 0.5:
            pair = torch.flip(pair, dims=[3])
        pair = (pair - self.mean) / self.std
        ef = float(row[self.ef_col])
        label = 1.0 if ef < HFrEF_THRESHOLD else 0.0
        quality = float(row['quality_score']) if not pd.isna(row['quality_score']) else 0.5
        return pair, torch.tensor(ef, dtype=torch.float32), torch.tensor(label, dtype=torch.float32), torch.tensor(quality, dtype=torch.float32)

BATCH_SIZE = 24
train_ds = EchoEDESDataset(train_df, EDES_CACHE_DIR, filename_col, ef_col, train=True)
val_ds = EchoEDESDataset(val_df, EDES_CACHE_DIR, filename_col, ef_col, train=False)
test_ds = EchoEDESDataset(test_df, EDES_CACHE_DIR, filename_col, ef_col, train=False)
# num_workers=0: avoids the flaky-Drive-under-concurrent-access failures seen with
# num_workers=2, and makes results fully reproducible across identical seeds (multi-worker
# DataLoaders have their own non-determinism even with set_seed()). Slower, but reliable --
# worth it for the ablation/seed-variance runs specifically.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


## Model definition (needed to instantiate the model before loading its checkpoint)

In [ ]:
class QualityFiLM(nn.Module):
    def __init__(self, feat_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, 2 * feat_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
        self.feat_dim = feat_dim

    def forward(self, quality_score):
        params = self.net(quality_score.unsqueeze(-1))
        gamma, beta = params[:, :self.feat_dim], params[:, self.feat_dim:]
        return 1.0 + gamma, beta

class EchoEDESCQRAblatable(nn.Module):
    """Same architecture as notebook 15's EchoEDESCQR, with use_delta / use_film
    toggles for the component-wise ablation ."""
    def __init__(self, dropout_p=0.2, pretrained_backbone_path=None,
                 use_delta=True, use_film=True):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()
        if pretrained_backbone_path:
            backbone.load_state_dict(torch.load(pretrained_backbone_path, map_location='cpu'))
        self.backbone = backbone
        self.use_delta = use_delta
        self.use_film = use_film
        combined_dim = 512 * 3 if use_delta else 512 * 2
        self.film = QualityFiLM(feat_dim=combined_dim) if use_film else None
        self.dropout = nn.Dropout(p=dropout_p)
        self.ef_mean_head = nn.Linear(combined_dim, 1)
        self.ef_q_lo_head = nn.Linear(combined_dim, 1)
        self.ef_q_hi_head = nn.Linear(combined_dim, 1)
        self.cls_head = nn.Linear(combined_dim, 1)

    def forward(self, pair, quality_score):
        ed = pair[:, 0]
        es = pair[:, 1]
        f_ed = self.backbone(ed)
        f_es = self.backbone(es)
        feats = torch.cat([f_ed, f_es, f_ed - f_es], dim=1) if self.use_delta else torch.cat([f_ed, f_es], dim=1)
        if self.use_film:
            gamma, beta = self.film(quality_score)
            feats = gamma * feats + beta
        feats = self.dropout(feats)
        ef_mean = self.ef_mean_head(feats).squeeze(-1)
        q_lo = self.ef_q_lo_head(feats).squeeze(-1)
        q_hi = self.ef_q_hi_head(feats).squeeze(-1)
        cls_logit = self.cls_head(feats).squeeze(-1)
        return ef_mean, q_lo, q_hi, cls_logit

def pinball_loss(y_true, y_pred, tau):
    diff = y_true - y_pred
    return torch.mean(torch.max(tau * diff, (tau - 1) * diff))

TAU_LO, TAU_HI = 0.05, 0.95

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def evaluate_simple(model, loader):
    model.eval()
    ef_true_all, ef_pred_all, cls_true_all, cls_prob_all = [], [], [], []
    with torch.no_grad():
        for pair, ef_true, cls_true, quality in loader:
            pair, quality = pair.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
            ef_true_all.extend(ef_true.numpy()); ef_pred_all.extend(ef_mean.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    mae = mean_absolute_error(ef_true_all, ef_pred_all)
    try:
        auc = roc_auc_score(cls_true_all, cls_prob_all)
    except ValueError:
        auc = float('nan')
    return {'mae': mae, 'auc': auc}

def predict_all(model, loader):
    model.eval()
    ef_true_all, ef_mean_all, q_lo_all, q_hi_all, cls_true_all, cls_prob_all = [], [], [], [], [], []
    with torch.no_grad():
        for pair, ef_true, cls_true, quality in loader:
            pair, quality = pair.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
            ef_true_all.extend(ef_true.numpy()); ef_mean_all.extend(ef_mean.cpu().numpy())
            q_lo_all.extend(q_lo.cpu().numpy()); q_hi_all.extend(q_hi.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    return {
        'ef_true': np.array(ef_true_all), 'ef_mean': np.array(ef_mean_all),
        'q_lo': np.array(q_lo_all), 'q_hi': np.array(q_hi_all),
        'cls_true': np.array(cls_true_all), 'cls_prob': np.array(cls_prob_all),
    }


## Risk-coverage table for the final model

Reloads the final model (contraction-delta + Quality-FiLM, SimCLR-initialized). Two uncertainty signals, matching Section 3.5 of the paper:
- **EF (regression):** CQR interval width `q_hi - q_lo` (narrower = more confident).
- **HFrEF (classification):** MC-Dropout predictive entropy, T=20 stochastic passes.

A sanity check compares the reproduced test MAE to the value reported in the paper (6.052); if it is far off, the checkpoint path is probably wrong.

In [ ]:
# Reload the exact final model (delta+FiLM, SimCLR-init) from the notebook-15 checkpoint.
assert os.path.exists(FINAL_MODEL_CKPT), \
    f'Checkpoint not found at {FINAL_MODEL_CKPT} -- update the path if notebook 15 saved elsewhere.'

final_model = EchoEDESCQRAblatable(pretrained_backbone_path=None, use_delta=True, use_film=True).to(device)
ckpt = torch.load(FINAL_MODEL_CKPT, map_location=device)
final_model.load_state_dict(ckpt['model_state'])
print(f'Loaded final model from {FINAL_MODEL_CKPT} (epoch {ckpt.get("epoch", "?")})')

# Recompute the CQR correction Q_hat exactly as in notebook 15 cell 9.
val_results = predict_all(final_model, val_loader)
rng = np.random.default_rng(42)
n_val = len(val_results['ef_true'])
perm = rng.permutation(n_val)
calib_idx, calibcheck_idx = perm[:n_val // 2], perm[n_val // 2:]
ALPHA = 0.1
E_calib = np.maximum(val_results['q_lo'][calib_idx] - val_results['ef_true'][calib_idx],
                      val_results['ef_true'][calib_idx] - val_results['q_hi'][calib_idx])
Q_hat = np.quantile(E_calib, min(1.0, (1 - ALPHA) * (1 + 1 / len(calib_idx))))
print(f'CQR correction Q_hat: {Q_hat:.3f}')

test_results = predict_all(final_model, test_loader)
print(f'Sanity check -- TEST EF MAE: {mean_absolute_error(test_results["ef_true"], test_results["ef_mean"]):.3f} '
      f'(paper: 6.052)')

def enable_mc_dropout(model):
    """eval() for BatchNorm etc., but keep Dropout layers stochastic."""
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

def mc_dropout_entropy(model, loader, T=20):
    enable_mc_dropout(model)
    all_probs = []  # T x N
    cls_true_all = None
    with torch.no_grad():
        for t in range(T):
            probs_t, true_t = [], []
            for pair, ef_true, cls_true, quality in loader:
                pair, quality = pair.to(device), quality.to(device)
                _, _, _, cls_logit = model(pair, quality)
                probs_t.extend(torch.sigmoid(cls_logit).cpu().numpy())
                true_t.extend(cls_true.numpy())
            all_probs.append(probs_t)
            if cls_true_all is None:
                cls_true_all = np.array(true_t)
    all_probs = np.array(all_probs)  # T x N
    p_bar = all_probs.mean(axis=0)
    eps = 1e-8
    entropy = -p_bar * np.log(p_bar + eps) - (1 - p_bar) * np.log(1 - p_bar + eps)
    return p_bar, entropy, cls_true_all

ef_width = test_results['q_hi'] - test_results['q_lo']
p_bar, entropy, cls_true_mc = mc_dropout_entropy(final_model, test_loader, T=20)

coverage_levels = [1.0, 0.8, 0.6, 0.4]
rows = []
n = len(test_results['ef_true'])

# EF: retain the (coverage * n) videos with the SMALLEST CQR interval width
ef_order = np.argsort(ef_width)
for cov in coverage_levels:
    k = int(round(cov * n))
    keep = ef_order[:k]
    mae_k = mean_absolute_error(test_results['ef_true'][keep], test_results['ef_mean'][keep])
    rows.append({'coverage': cov, 'metric': 'EF MAE', 'value': mae_k, 'n_retained': k})

# HFrEF: retain the (coverage * n) videos with the LOWEST predictive entropy
cls_order = np.argsort(entropy)
for cov in coverage_levels:
    k = int(round(cov * n))
    keep = cls_order[:k]
    try:
        auc_k = roc_auc_score(cls_true_mc[keep], p_bar[keep])
    except ValueError:
        auc_k = float('nan')
    rows.append({'coverage': cov, 'metric': 'HFrEF AUC', 'value': auc_k, 'n_retained': k})

risk_coverage_df = pd.DataFrame(rows)
print(risk_coverage_df.to_string(index=False))

risk_coverage_df.to_csv(os.path.join(PROJECT_ROOT, 'logs', 'final_model_risk_coverage.csv'), index=False)
print('\nSaved to logs/final_model_risk_coverage.csv -- these are the Table 4 values (Section 4.5).')